In [2]:
import time
import hmac
import hashlib
import requests
import os
from dotenv import load_dotenv
from urllib.parse import urlencode

load_dotenv()

# 1. Set up your API Key and Secret
BASE_URL = "https://mock-api.roostoo.com"
API_KEY = os.getenv("TESTING_API_KEY")
SECRET_KEY = os.getenv("TESTING_API_SECRET")

In [3]:
# 2. Define the Base URL and Endpoint
# (Update the base URL to match Roostoo's specific environment if needed)
ENDPOINT = '/v3/exchangeInfo'


def _get_timestamp():
    """Return a 13-digit millisecond timestamp as string."""
    return str(int(time.time() * 1000))


def _get_signed_headers(payload: dict = {}):
    """
    Generate signed headers and totalParams for RCL_TopLevelCheck endpoints.
    """
    payload['timestamp'] = _get_timestamp()
    sorted_keys = sorted(payload.keys())
    total_params = "&".join(f"{k}={payload[k]}" for k in sorted_keys)

    signature = hmac.new(
        SECRET_KEY.encode('utf-8'),
        total_params.encode('utf-8'),
        hashlib.sha256
    ).hexdigest()

    headers = {
        'RST-API-KEY': API_KEY,
        'MSG-SIGNATURE': signature
    }

    return headers, payload, total_params

In [4]:
# ------------------------------
# Public Endpoints
# ------------------------------

def check_server_time():
    """Check API server time."""
    url = f"{BASE_URL}/v3/serverTime"
    try:
        res = requests.get(url)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error checking server time: {e}")
        return None


def get_exchange_info():
    """Get exchange trading pairs and info."""
    url = f"{BASE_URL}/v3/exchangeInfo"
    try:
        res = requests.get(url)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error getting exchange info: {e}")
        return None


def get_ticker(pair=None):
    """Get ticker for one or all pairs."""
    url = f"{BASE_URL}/v3/ticker"
    params = {'timestamp': _get_timestamp()}
    if pair:
        params['pair'] = pair
    try:
        res = requests.get(url, params=params)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error getting ticker: {e}")
        return None


# ------------------------------
# Signed Endpoints
# ------------------------------

def get_balance():
    """Get wallet balances (RCL_TopLevelCheck)."""
    url = f"{BASE_URL}/v3/balance"
    headers, payload, _ = _get_signed_headers({})
    try:
        res = requests.get(url, headers=headers, params=payload)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error getting balance: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None


def get_pending_count():
    """Get total pending order count."""
    url = f"{BASE_URL}/v3/pending_count"
    headers, payload, _ = _get_signed_headers({})
    try:
        res = requests.get(url, headers=headers, params=payload)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error getting pending count: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None


def place_order(pair_or_coin, side, quantity, price=None, order_type=None):
    """
    Place a LIMIT or MARKET order.
    """
    url = f"{BASE_URL}/v3/place_order"
    pair = f"{pair_or_coin}/USD" if "/" not in pair_or_coin else pair_or_coin

    if order_type is None:
        order_type = "LIMIT" if price is not None else "MARKET"

    if order_type == 'LIMIT' and price is None:
        print("Error: LIMIT orders require 'price'.")
        return None

    payload = {
        'pair': pair,
        'side': side.upper(),
        'type': order_type.upper(),
        'quantity': str(quantity)
    }
    if order_type == 'LIMIT':
        payload['price'] = str(price)

    headers, _, total_params = _get_signed_headers(payload)
    headers['Content-Type'] = 'application/x-www-form-urlencoded'

    try:
        res = requests.post(url, headers=headers, data=total_params)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error placing order: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None


def query_order(order_id=None, pair=None, pending_only=None):
    """Query order history or pending orders."""
    url = f"{BASE_URL}/v3/query_order"
    payload = {}
    if order_id:
        payload['order_id'] = str(order_id)
    elif pair:
        payload['pair'] = pair
        if pending_only is not None:
            payload['pending_only'] = 'TRUE' if pending_only else 'FALSE'

    headers, _, total_params = _get_signed_headers(payload)
    headers['Content-Type'] = 'application/x-www-form-urlencoded'

    try:
        res = requests.post(url, headers=headers, data=total_params)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error querying order: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None


def cancel_order(order_id=None, pair=None):
    """Cancel specific or all pending orders."""
    url = f"{BASE_URL}/v3/cancel_order"
    payload = {}
    if order_id:
        payload['order_id'] = str(order_id)
    elif pair:
        payload['pair'] = pair

    headers, _, total_params = _get_signed_headers(payload)
    headers['Content-Type'] = 'application/x-www-form-urlencoded'

    try:
        res = requests.post(url, headers=headers, data=total_params)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error canceling order: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None

In [14]:
print("\n--- Checking Server Time ---")
print(check_server_time())

print("\n--- Getting Exchange Info ---")
info = get_exchange_info()
print(info)

print("\n--- Getting Market Ticker (BTC/USD) ---")
ticker = get_ticker("BTC/USD")
if ticker:
    print(ticker.get("Data", {}).get("BTC/USD", {}))


--- Checking Server Time ---
{'ServerTime': 1773736755224}

--- Getting Exchange Info ---
{'IsRunning': True, 'InitialWallet': {'USD': 50000}, 'TradePairs': {'OPEN/USD': {'Coin': 'OPEN', 'CoinFullName': 'OPEN', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 4, 'AmountPrecision': 1, 'MiniOrder': 1}, 'TRUMP/USD': {'Coin': 'TRUMP', 'CoinFullName': 'TRUMP', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 3, 'AmountPrecision': 3, 'MiniOrder': 1}, 'TON/USD': {'Coin': 'TON', 'CoinFullName': 'Toncoin', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 3, 'AmountPrecision': 2, 'MiniOrder': 1}, 'S/USD': {'Coin': 'S', 'CoinFullName': 'S', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 5, 'AmountPrecision': 1, 'MiniOrder': 1}, 'SOL/USD': {'Coin': 'SOL', 'CoinFullName': 'SOL', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 2, 'AmountPrecis

In [13]:
ticker = get_ticker("BNB/USD")
print(ticker.get("Data", {}).get("BNB/USD", {}))
print(place_order("BNB/USD", "BUY", 0.6116845008481413))      

{'MaxBid': 674.25, 'MinAsk': 674.26, 'LastPrice': 674.26, 'Change': -0.0036, 'CoinTradeValue': 214901.259, 'UnitTradeValue': 145709690.67213}
{'Success': False, 'ErrMsg': 'quantity step size error'}


In [14]:
print(get_pending_count())

{'Success': False, 'ErrMsg': 'no pending order under this account', 'TotalPending': 0, 'OrderPairs': {}}


In [17]:
print(get_balance())

{'Success': True, 'ErrMsg': '', 'SpotWallet': {'BNB': {'Free': 0, 'Lock': 0}, 'BTC': {'Free': 0, 'Lock': 0}, 'USD': {'Free': 49884.23, 'Lock': 0}}, 'MarginWallet': {}}


In [12]:
print(ticker.get("Data", {}).get("BNB/USD", {}))

{'MaxBid': 674.74, 'MinAsk': 674.75, 'LastPrice': 674.75, 'Change': -0.0026, 'CoinTradeValue': 215498.46, 'UnitTradeValue': 146113916.67455}


In [16]:
print(place_order("BTC/USD", "SELL", 0.61078))

{'Success': True, 'ErrMsg': '', 'OrderDetail': {'Pair': 'BTC/USD', 'OrderID': 2760878, 'Status': 'FILLED', 'Role': 'TAKER', 'ServerTimeUsage': 0.00457181, 'CreateTimestamp': 1773737308323, 'FinishTimestamp': 1773737308327, 'Side': 'SELL', 'Type': 'MARKET', 'StopType': 'GTC', 'Price': 74392.92, 'Quantity': 0.61078, 'FilledQuantity': 0.61078, 'FilledAverPrice': 74392.92, 'CoinChange': 0.61078, 'UnitChange': 45437.707677599996, 'CommissionCoin': 'USD', 'CommissionChargeValue': 45.437707, 'CommissionPercent': 0.001, 'OrderWalletType': 'SPOT', 'OrderSource': 'PUBLIC_API'}}
